#I. Data set description

| Variable | Type | Description | Units |
|:---------|:-----|:------------|:------|
| `id_lab` | Character | Laboratory sample identifier | — |
| `id` | Character | Germplasm accession identifier | — |
| `subset` | Integer | Experimental subset identifier | — |
| `no` | Numeric | Sample number | — |
| `requisitioner` | Character | Source or requesting program of the accession | — |
| `tax_name` | Character | Scientific (taxonomic) name of the species | — |
| `functional_group` | Character | Functional group classification | — |
| `n_replicates_nutrition` | Integer | Number of replicates used for nutritional analyses | Replicates |
| `dm_percentage` | Numeric | Dry matter (DM) content | % |
| `ash_dm` | Numeric | Ash content on a dry matter basis | % DM |
| `om_percentage` | Numeric | Organic matter content | % |
| `pc_percentage_dm` | Numeric | Crude protein (CP) content on a dry matter basis | % DM |
| `adf_percentage_dm` | Numeric | Acid detergent fiber (ADF) content on a dry matter basis | % DM |
| `ndf_percentage_dm` | Numeric | Neutral detergent fiber (NDF) content on a dry matter basis | % DM |
| `n_replicates_gas` | Integer | Number of replicates used for gas production measurements | Replicates |
| `ch4_percentage_in_gas_8h` | Numeric | Methane concentration in fermentation gas after 8 h of incubation | % |
| `ch4_percentage_in_gas_24h` | Numeric | Methane concentration in fermentation gas after 24 h of incubation | % |
| `methane_intensity` | Numeric | Methane intensity measured during fermentation | *(units depend on calculation protocol)* |
| `tddm` | Numeric | True dry matter digestibility (TDDM) | % |

# 1.0 Install packages and import libraries

In [1]:
install.packages(c("dplyr","lme4","emmeans"))

Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘rbibutils’, ‘Rdpack’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘RcppEigen’, ‘estimability’, ‘mvtnorm’, ‘numDeriv’




In [2]:
library(dplyr)
library(lme4)
library(emmeans)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: Matrix

Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'



# 2.0 Data load

In [3]:
url_gas_clean <- "https://raw.githubusercontent.com/maurope/lmf/main/output/2026_06_09_trial_database_curation/gas_clean_complete_subsets_1234_2026_06_09.csv"
df <- read.csv(url_gas_clean)

In [4]:
head(df)

,subset,no,requisitioner,id_lab,id,tax_order,family,genus,species,tax_name,⋯,ch4_ml_g_ndf_digested_24h,methane_intensity,tddm,information_remarks_1,information_remarks_2,gas_remarks_1,gas_remarks_2,digest_remarks_1,digest_remarks_2,delete
,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,⋯,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<lgl>,<chr>,<chr>,<chr>
1,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,⋯,0,95.43754,34.82974,Standar,,,NA,,,no
2,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,⋯,0,89.75201,36.19470,Standar,,,NA,,,no
3,1,#REF!,LMF,F25-0017,Hohenheimer-Heustandard,Forage,Forage,Hohenheimer,Heustandard,Hohenheimer Heustandard,⋯,0,59.21398,56.92077,Standar,,,NA,,,no
4,1,#REF!,Genetic_bank,F25-0017,Hohenheimer-Heustandard,Fabales,Fabaceae,Hohenheimer,Heustandard,Hohenheimer Heustandard,⋯,0,95.43754,34.82974,Standar,,,NA,,,no
5,1,#REF!,Genetic_bank,F25-0017,Hohenheimer-Heustandard,Fabales,Fabaceae,Hohenheimer,Heustandard,Hohenheimer Heustandard,⋯,0,89.75201,36.19470,Standar,,,NA,,,no
6,1,#REF!,Genetic_bank,F25-0017,Hohenheimer-Heustandard,Fabales,Fabaceae,Hohenheimer,Heustandard,Hohenheimer Heustandard,⋯,0,59.21398,56.92077,Standar,,,NA,,,no


# 3.0 Mixed Models to BLUEs estimation

In [5]:
colnames(df)

[1] "subset"                    "no"                       
 [3] "requisitioner"             "id_lab"                   
 [5] "id"                        "tax_order"                
 [7] "family"                    "genus"                    
 [9] "species"                   "tax_name"                 
[11] "functional_group"          "set_ciat"                 
[13] "batch"                     "run"                      
[15] "replication"               "syrange"                  
[17] "sample_weight_g"           "undigested_dm_g"          
[19] "dm_incubated"              "digested_feed_mg"         
[21] "net_gas_8h_ml"             "net_gas_24h_ml"           
[23] "ch4_8h_ml"                 "ch4_24h_ml"               
[25] "ch4_percentage_in_gas_8h"  "ch4_percentage_in_gas_24h"
[27] "gas_ml_g_dm_incubated_24h" "part_fact"                
[29] "ch4_ml_g_dm_incubated_24h" "ch4_ml_g_ndf_digested_24h"
[31] "methane_intensity"         "tddm"                     
[33] "information_remarks_1"     "information_remarks_2"    
[35] "gas_remarks_1"             "gas_remarks_2"            
[37] "digest_remarks_1"          "digest_remarks_2"         
[39] "delete"

original script sent by Khaled

In [7]:
df$id_lab  <- factor(df$id_lab)
df$batch   <- factor(df$batch)

traits <- colnames(df)[17:32]

blues <- list()
blups <- list()

write("Analysis Report:", file = "analysis_report.txt")

for (trait in traits) {
  model.1 <- lm(formula(paste0("`", trait, "` ~ id_lab + batch")), data = df)

  write("\n========================\n", file = "analysis_report.txt", append = TRUE)
  capture.output(anova(model.1), file = "analysis_report.txt", append = TRUE)

  model.2 <- lmer(formula(paste0("`", trait, "` ~ (1 | id_lab) + (1 | batch)")), data = df, REML = TRUE)

  vc <- as.data.frame(VarCorr(model.2))
  vc$variation_source <- vc$grp
  vc$variance_ratio <- sprintf("%.2f%%", 100 * vc$vcov / sum(vc$vcov))

  write("\nVariance components for random effects:", file = "analysis_report.txt", append = TRUE)
  capture.output(vc[,c("variation_source", "variance_ratio")], file = "analysis_report.txt", append = TRUE)

  ranef_info <- ranef(model.2, condVar = TRUE)
  fixed_eff  <- fixef(model.2)["(Intercept)"]
  blups_eff  <- ranef_info$id_lab
  blups_se   <- sqrt(attr(ranef_info$id_lab, "postVar")[1, , ])
  blups_df   <- data.frame(id_lab = rownames(blups_eff), BLUP = blups_eff$`(Intercept)` + fixed_eff, SE = blups_se)

  colnames(blups_df) <- c("id_lab", trait, paste0(trait, "_SE"))
  blups[[trait]] <- blups_df

  model.3 <- lmer(formula(paste0("`", trait, "` ~ id_lab + (1 | batch)")), data = df, REML = TRUE)

  blues_emm <- emmeans(model.3, ~ id_lab, df = Inf)  # Here I modified the df to Inf because the default limit of the package is 3000 observations and here I analyze more than 6000
  blues_df  <- summary(blues_emm)

  pairwise_lsd <- as.data.frame(summary(pairs(blues_emm, adjust = "none")))
  pairwise_lsd$LSD <- qt(0.975, df = pairwise_lsd$df) * pairwise_lsd$SE

  avg_LSD <- signif(mean(pairwise_lsd$LSD, na.rm = TRUE), 4)
  write(paste("\nAverage LSD:", avg_LSD), file = "analysis_report.txt", append = TRUE)

  blues_df <- blues_df[, c("id_lab", "emmean", "SE")]
  colnames(blues_df) <- c("id_lab", trait, paste0(trait, "_SE"))

  blues[[trait]] <- blues_df
}

blue <- Reduce(function(x, y) merge(x, y, by = names(x)[1], all = TRUE), blues)
blup <- Reduce(function(x, y) merge(x, y, by = names(x)[1], all = TRUE), blups)

write.csv(blue, file = "blues.csv", row.names = FALSE)
write.csv(blup, file = "blups.csv", row.names = FALSE)


Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'pbkrtest.limit = 6112' (or larger)
[or, globally, 'set emm_options(pbkrtest.limit = 6112)' or larger];
but be warned that this may result in large computation time and memory use.

Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'lmerTest.limit = 6112' (or larger)
[or, globally, 'set emm_options(lmerTest.limit = 6112)' or larger];
but be warned that this may result in large computation time and memory use.



Modified script just estimating blues from methane intensity and tddm

In [ ]:
library(lme4)
library(emmeans)

# Convert grouping variables to factors
df$id_lab <- factor(df$id_lab)
df$batch  <- factor(df$batch)

# Traits to analyze
traits <- colnames(df)[31:32]

# Store BLUEs
blues <- list()

# Analysis report
write("BLUE Analysis Report:", file = "analysis_report.txt")

for (trait in traits) {

  # Mixed model:
  # id_lab = fixed effect
  # batch  = random effect
  model <- lmer(
    formula(paste0("`", trait, "` ~ id_lab + (1 | batch)")),
    data = df,
    REML = TRUE
  )

  # Estimated marginal means (BLUEs)
  blues_emm <- emmeans(
    model,
    ~ id_lab,
    df = Inf
  )

  blues_df <- as.data.frame(summary(blues_emm))

  # Keep only BLUE and SE
  blues_df <- blues_df[, c("id_lab", "emmean", "SE")]

  # Rename columns
  colnames(blues_df) <- c(
    "id_lab",
    trait,
    paste0(trait, "_SE")
  )

  # Store BLUEs
  blues[[trait]] <- blues_df

  # Calculate pairwise LSD
  pairwise_lsd <- as.data.frame(
    summary(pairs(blues_emm, adjust = "none"))
  )

  pairwise_lsd$LSD <- qt(
    0.975,
    df = pairwise_lsd$df
  ) * pairwise_lsd$SE

  avg_LSD <- signif(
    mean(pairwise_lsd$LSD, na.rm = TRUE),
    4
  )

  # Write information to report
  write(
    paste("\n========================\nTrait:", trait),
    file = "analysis_report.txt",
    append = TRUE
  )

  write(
    paste("Average LSD:", avg_LSD),
    file = "analysis_report.txt",
    append = TRUE
  )
}

# Merge BLUEs for all traits
blue <- Reduce(
  function(x, y) merge(
    x,
    y,
    by = "id_lab",
    all = TRUE
  ),
  blues
)

# Save BLUEs
write.csv(
  blue,
  file = "blues.csv",
  row.names = FALSE
)

Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'pbkrtest.limit = 6486' (or larger)
[or, globally, 'set emm_options(pbkrtest.limit = 6486)' or larger];
but be warned that this may result in large computation time and memory use.

Note: D.f. calculations have been disabled because the number of observations exceeds 3000.
To enable adjustments, add the argument 'lmerTest.limit = 6486' (or larger)
[or, globally, 'set emm_options(lmerTest.limit = 6486)' or larger];
but be warned that this may result in large computation time and memory use.

